In [1]:
# Current path:
!pwd # Current repo - "/Users/pini/wix_home_assignment"

/Users/pini/wix_home_assignment


# Nexa Treasury Case - Technical Appendix

This repo has the code behind the Nexa Treasury case study.

The case asks three questions about 2 March to 10 April 2026:

| | Question | Notebook |
| --- | --- | --- |
| A | Where is our cash, and does it reconcile to what we expected? | `task_a1_a2_a3_cash_position_n_reconciliation.ipynb` |
| B | How much FX exposure do we have, and what should we hedge? | `task_b1_b2_fx_expo_n_hedging.ipynb` |
| C | How healthy are our collections, and how do we answer the AR emails? | `task_c1_c2_c3_collections_n_ar_process.ipynb` |

## What is in the folders

| Path | What it is |
| --- | --- |
| `source_files/` | The raw input data from the case (bank report, expected collections, FX rates, subscription forecast, AR emails). Not modified. |
| `analytics_output/` | CSV files written by the notebooks. The dashboard reads these. |
| `db_visual_streamlit.py` | The dashboard (Streamlit). |
| `datasets_spec/` | The data specification that came with the case. |

## How to run it

**1. Install what you need** (Python 3.9 or newer):

```bash
pip install pandas "numpy<2" streamlit plotly jupyter
```

**2. Run the notebooks in this order** (open each one in Jupyter or VS Code and choose *Run All*):

1. Notebook A
2. Notebook B
3. Notebook C

Order matters only for C, which reads a file that A writes (`reconciliation_detail.csv`). Each notebook saves its results to `analytics_output/` in its last cell.

**3. Start the dashboard:**

```bash
streamlit run db_visual_streamlit.py
```

Then open the link Streamlit prints (usually http://localhost:8501). Use the menu on the left to switch between the overview and tasks A, B and C.


## Where to find the reasoning

Each notebook has short text cells that explain the method, the assumptions and what was fixed in the data. Start there if you want to understand a number. The main decisions are:

* **Cash (A):** balances use the *value date* and the daily mid rate for each date.
* **Reconciliation (A):** each bank receipt is matched to one expected item. Small gaps (under 0.5%) count as fees. Anything else is flagged with an owner and a next step.
* **FX (B):** exposure is collections minus refunds minus local costs, using the 10 April spot. Hedge sizes follow the policy ranges in the case.
* **AR (C):** the aging covers open customer invoices only, after taking out what has already been paid. Email decisions follow the credit policy and were checked against the bank data.

## Outputs at a glance

| File in `analytics_output/` | Written by | What it shows |
| --- | --- | --- |
| `cash_positions_summary.csv` | A | Cash by account and currency, local and USD, for 31 Mar and 10 Apr |
| `reconciliation_status_counts.csv`, `reconciliation_detail.csv` | A | Matched, timing difference or exception for every receipt |
| `detailed_exceptions_with_owners.csv` | A | Each exception with owner and next step |
| `timing_differences_summary.csv`, `exceptions_summary.csv` | A | Totals by entity, bank and currency |
| `forecast_net_fx_exposure_summary.csv`, `fx_monthly_exposure.csv` | B | FX exposure by currency and by month |
| `fx_rates_forward_analysis.csv`, `fx_hedging_recommendation.csv` | B | Forward rates and the hedging recommendation |
| `ar_aging_detailed.csv`, `ar_aging_by_bucket_currency.csv` | C | Open invoices and the aging |
| `ar_segment_performance.csv`, `ar_process_kpis.csv` | C | Collections health by segment and key process metrics |
| `ar_email_decision_matrix.csv` | C | Recommended response to each of the six emails |

## If something goes wrong

* **NumPy or pandas import errors:** install `numpy<2`, as in the command above.
* **Dashboard says a file is missing:** run the notebooks first (A, then B, then C).
* **Notebook cannot find a file:** open the notebook from this folder, so the relative paths work.


# Nexa Treasury Case - python notebook - task C

In [2]:
# In this part, I made sure to use simple but effective python data analysis
# approaches in order to keep things simple, clean and easy to debug/examine (for the sake of this home assignment)

# Relevant libs for this project:
import pandas as pd
import numpy as np

import os

/Users/pini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/pini/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


### Data sources

In [3]:
# Datasets
df_ar = pd.read_csv('source_files/expected_collections.csv')
df_fx = pd.read_csv('source_files/fx_rates.csv')

# Output of notebook A: receipts matched to invoices (run notebook A before this one)
df_recon = pd.read_csv('analytics_output/reconciliation_detail.csv')

In [4]:
# Prepare column formats
df_ar.columns = [x.replace(' ', '_') for x in df_ar.columns]
df_ar.columns = [x.lower() for x in df_ar.columns]

### Task C1 - Build an AR aging as of 10 April 2026,

In [5]:
# Convert date columns to datetime format
df_ar['invoice_date'] = pd.to_datetime(df_ar['invoice_date'])
df_ar['service_period_start'] = pd.to_datetime(df_ar['service_period_start'])
df_ar['due_date'] = pd.to_datetime(df_ar['due_date'])

# Define snapshot date which is: 10 April 2026
snapshot_date = pd.to_datetime('2026-04-10')

# Customer invoices only - PSP payouts are expected receipts, not open receivables
df_ar = df_ar[df_ar['source_type'] == 'Invoice'].copy()

# How much was already received per invoice (from notebook A); a wire returned by the bank does not count as paid
paid_receipts = df_recon[df_recon['expected_id'].astype(str).str.startswith('INV-')
                         & (df_recon['exception_reason'] != 'Receipt returned by bank (wire return)')]
df_ar['received_amount'] = df_ar['collection_id'].map(paid_receipts.groupby('expected_id')['amount'].sum()).fillna(0.0)
df_ar['open_amount'] = (df_ar['amount_due'] - df_ar['received_amount']).clip(lower=0)

# A residual under 0.5% (bank fees) counts as closed - same tolerance as notebook A
df_ar.loc[df_ar['open_amount'] < df_ar['amount_due'] * 0.005, 'open_amount'] = 0.0

# Convert to USD at the latest spot rate (10 April 2026)
df_fx_latest = df_fx[df_fx['rate_date'] == df_fx['rate_date'].max()].set_index('currency')

def to_usd(amount, currency):
    if currency == 'USD':
        return amount
    fx = df_fx_latest.loc[currency]
    return amount * fx['spot_mid'] if 'USD per 1' in fx['quote_convention'] else amount / fx['spot_mid']

df_ar['amount_due_usd'] = df_ar.apply(lambda r: to_usd(r['amount_due'], r['currency']), axis=1)
df_ar['open_amount_usd'] = df_ar.apply(lambda r: to_usd(r['open_amount'], r['currency']), axis=1)

df_ar['days_overdue'] = (snapshot_date - df_ar['due_date']).dt.days

# 3. Aging Buckets 
def assign_aging_bucket(days):
    if days <= 0:
        return '01. Not Due'
    elif days <= 30:
        return '02. 1-30 Days'
    elif days <= 60:
        return '03. 31-60 Days'
    elif days <= 90:
        return '04. 61-90 Days'
    else:
        return '05. 90+ Days'

df_ar['aging_bucket'] = df_ar['days_overdue'].apply(assign_aging_bucket)

# Open receivables only (balance > 0) form the aging base
df_open = df_ar[df_ar['open_amount'] > 0].copy()

print(f"Invoices in file: {len(df_ar)} | still open at 10 Apr: {len(df_open)} | paid / closed: {len(df_ar) - len(df_open)}")

# Aging by bucket and currency (amounts in USD)
ar_aging_summary = df_open.groupby(['aging_bucket', 'currency']).agg(
    invoice_count=('collection_id', 'count'),
    total_amount_usd=('open_amount_usd', 'sum')
).reset_index()

print("AR Aging Summary as of 10 April 2026 (USD)")
pd.options.display.float_format = '{:,.2f}'.format
# Presentation
display(df_open.groupby('aging_bucket').agg(
    invoice_count=('collection_id', 'count'), total_amount_usd=('open_amount_usd', 'sum')).reset_index())

print("Aging by segment (USD)")
display(df_open.pivot_table(index='segment', columns='aging_bucket', values='open_amount_usd', aggfunc='sum', fill_value=0))

print("Top 10 largest overdue items (USD)")
display(df_open[df_open['days_overdue'] > 0].sort_values('open_amount_usd', ascending=False).head(10)[
    ['collection_id', 'customer_or_psp', 'segment', 'currency', 'open_amount', 'open_amount_usd', 'days_overdue', 'sales_owner']])

Invoices in file: 101 | still open at 10 Apr: 50 | paid / closed: 51
AR Aging Summary as of 10 April 2026 (USD)


,aging_bucket,invoice_count,total_amount_usd
0,01. Not Due,28,"1,247,722.13"
1,02. 1-30 Days,11,"783,635.24"
2,03. 31-60 Days,1,"18,750.00"
3,04. 61-90 Days,1,"18,750.00"
4,05. 90+ Days,9,"372,805.51"


Aging by segment (USD)


aging_bucket,01. Not Due,02. 1-30 Days,03. 31-60 Days,04. 61-90 Days,05. 90+ Days
segment,,,,,
Enterprise,"1,071,995.01","701,588.42",0.00,0.00,"262,389.00"
Mid-Market,"175,727.13","82,046.82","18,750.00","18,750.00","110,416.51"


Top 10 largest overdue items (USD)


,collection_id,customer_or_psp,segment,currency,open_amount,open_amount_usd,days_overdue,sales_owner
119,INV-19644,Grupo Varanda SA,Enterprise,BRL,"1,850,000.00","302,604.03",29,Lucas Ferreira
8,INV-19510,Castello Foods SpA,Enterprise,EUR,"132,500.00","143,789.00",91,Sofia Brandt
198,INV-19591,Castello Foods SpA,Enterprise,EUR,"132,500.00","143,789.00",21,Sofia Brandt
202,INV-19606,Harborview Clinics,Enterprise,USD,"118,600.00","118,600.00",21,Megan Liu
3,INV-19499,Harborview Clinics,Enterprise,USD,"118,600.00","118,600.00",112,Megan Liu
288,INV-19726,Rede Palmeira SA,Enterprise,BRL,"615,000.00","100,595.39",9,Lucas Ferreira
191,INV-19666,Brightline Logistics,Enterprise,USD,"36,000.00","36,000.00",22,Jordan Pell
324,INV-19762,Silverline Events,Mid-Market,USD,"20,090.00","20,090.00",8,Aaron Whitfield
10,INV-19536,Beacon Fitness Clubs,Mid-Market,USD,"18,750.00","18,750.00",68,Aaron Whitfield
2,INV-19492,Beacon Fitness Clubs,Mid-Market,USD,"18,750.00","18,750.00",128,Aaron Whitfield


### Task C2 - Short process-efficiency view.

In [6]:
import pandas as pd
import numpy as np

# 1. Actual credit term and overdue status as of the snapshot date
df_ar['credit_term_days'] = (df_ar['due_date'] - df_ar['invoice_date']).dt.days
df_ar['is_overdue'] = df_ar['days_overdue'] > 0
df_open['is_overdue'] = df_open['days_overdue'] > 0

# 2. Invoicing speed: days from service start to invoice date (policy: invoice right at the billing trigger)
df_ar['invoice_lag_days'] = (df_ar['invoice_date'] - df_ar['service_period_start']).dt.days

# 3. Payment behaviour: days between the actual value date and the expected date, paid invoices only
paid_invoices = paid_receipts.merge(df_ar[['collection_id', 'segment']], left_on='expected_id', right_on='collection_id')

# 4. Unapplied cash and returned wires
unapplied = df_recon[df_recon['exception_reason'] == 'Unapplied receipt (no expected record)'].copy()
returned = df_recon[df_recon['exception_reason'] == 'Receipt returned by bank (wire return)']
unapplied['amount_usd'] = unapplied.apply(lambda r: to_usd(r['amount'], r['currency']), axis=1)

open_usd = df_open['open_amount_usd'].sum()
overdue_usd = df_open[df_open['is_overdue']]['open_amount_usd'].sum()

print("=== Process Efficiency & AR Metrics ===")
print(f"Open invoices: {len(df_open):,} | Open AR: ${open_usd:,.0f}")
print(f"Overdue share of open AR: {overdue_usd / open_usd * 100:.1f}% (${overdue_usd:,.0f})")
print(f"Average invoice lag after service start: {df_ar['invoice_lag_days'].mean():.1f} days "
      f"| issued more than 7 days late: {(df_ar['invoice_lag_days'] > 7).mean() * 100:.1f}% of invoices")
print(f"Paid invoices: {len(paid_invoices)} | avg days vs expected date: {paid_invoices['days_diff'].mean():.1f} "
      f"| paid more than 3 days late: {(paid_invoices['days_diff'] > 3).mean() * 100:.1f}%")
print(f"Unapplied receipts: {len(unapplied)} (${unapplied['amount_usd'].sum():,.0f}) | returned wires: {len(returned)}\n")

# 5. Performance by segment
segment_performance = df_open.groupby('segment').agg(
    open_amount_usd=('open_amount_usd', 'sum'),
    invoice_count=('collection_id', 'count'),
    avg_days_overdue=('days_overdue', lambda x: x[x > 0].mean()),
    overdue_share=('is_overdue', lambda x: x.mean() * 100)
).reset_index().fillna(0)

segment_performance['avg_invoice_lag_days'] = segment_performance['segment'].map(df_ar.groupby('segment')['invoice_lag_days'].mean())
segment_performance['avg_days_vs_expected_paid'] = segment_performance['segment'].map(paid_invoices.groupby('segment')['days_diff'].mean())

print("--- Performance by Segment ---")
pd.options.display.float_format = '{:,.2f}'.format
display(segment_performance)

=== Process Efficiency & AR Metrics ===
Open invoices: 50 | Open AR: $2,441,663
Overdue share of open AR: 48.9% ($1,193,941)
Average invoice lag after service start: 3.8 days | issued more than 7 days late: 10.9% of invoices
Paid invoices: 52 | avg days vs expected date: 3.2 | paid more than 3 days late: 30.8%
Unapplied receipts: 3 ($95,993) | returned wires: 1

--- Performance by Segment ---


,segment,open_amount_usd,invoice_count,avg_days_overdue,overdue_share,avg_invoice_lag_days,avg_days_vs_expected_paid
0,Enterprise,"2,035,972.43",17,43.57,41.18,3.51,2.11
1,Mid-Market,"405,690.46",33,66.67,45.45,4.00,3.88


### Task C3 - Response for six given emails (AR based)

*In a README.md format*

Here is a quick summary of what I recommend for the **6 emails**. Each one is checked against the bank report and the AR data in the code cell below (see the `data_check` column).

---

### **Summary of Email Decisions (Appendix A Compliance)**

| Email / Client | Action | Owner | Why (Appendix A) |
| --- | --- | --- | --- |
| **#1 Kestrel Media** *(USD 2.4M prepay)* | **Approve** | AR / Cash Application | The wire arrived on 26 Mar (it only quotes the PO) and equals INV-19810. Apply it by hand and send the paid receipt. |
| **#2 VP Sales - Net 90** | **Decline** | Director of Treasury | Net 90 is beyond Net 30 / 45 / 60 (policy 1). A blanket change needs written approval from the Director of Treasury (3b). |
| **#3 Lumen Retail** *(USD 96K)* | **Decline backdating** | Billing Ops + Collections Lead | Net 30 runs from the real invoice date, 16 Mar (policy 1 and 2). Fix the missing-PO delay at the source. |
| **#4 Grupo Varanda** *(BRL 1.85M)* | **Escalate (suspected fraud)** | Treasury + Security | Only Treasury issues bank details (policy 5). The bank returned the wire. Do not mark it paid or waive the fee (policy 4). |
| **#5 Tidewater** *(USD 38.4K)* | **Approve** | Collections Lead | One-time 30-day extension on a single invoice, and no other invoice is over 30 days late (policy 3a). |
| **#6 Unapplied cash** *(3 receipts)* | **Decline** | AR + Sales Ops | The money is in the bank report but has no invoice. Keep it in suspense and never book it to miscellaneous income. |

In [7]:
# Check every email against the bank report and AR data
kestrel = df_recon[df_recon['expected_id'] == 'INV-19810'].iloc[0]
varanda_row = df_recon[df_recon['expected_id'] == 'INV-19644'].iloc[0]
varanda_inv = df_ar[df_ar['collection_id'] == 'INV-19644'].iloc[0]
lumen_inv = df_ar[df_ar['collection_id'] == 'INV-19818'].iloc[0]
tidewater_inv = df_ar[df_ar['collection_id'] == 'INV-19825'].iloc[0]

# Condition 3(a): the customer has no other open invoice more than 30 days overdue
tidewater_other = df_open[df_open['customer_or_psp'].str.startswith('Tidewater')
                          & (df_open['collection_id'] != 'INV-19825') & (df_open['days_overdue'] > 30)]

# Total Enterprise invoices issued in March 2026 - the volume the Net 90 request would affect
march_enterprise_usd = df_ar[(df_ar['segment'] == 'Enterprise') & (df_ar['invoice_date'].dt.strftime('%Y-%m') == '2026-03')]['amount_due_usd'].sum()

lumen_due = lumen_inv['invoice_date'] + pd.Timedelta(days=30)
tidewater_new_due = tidewater_inv['due_date'] + pd.Timedelta(days=30)

email_data = [
    {"email_id": "Email #1", "client": "Kestrel Media", "segment": "Enterprise", "amount": "USD 2,400,000",
     "recommended_action": "Approve", "responsible_owner": "AR / Cash Application",
     "policy_justification": "Apply the wire to INV-19810 manually (reference is the PO only) and send a paid receipt. Confirm the remitter is the customer first.",
     "data_check": f"Bank report: wire of {kestrel['amount']:,.0f} {kestrel['currency']} on {kestrel['value_date'][:10]} (matched by customer + amount to {kestrel['expected_id']})."},
    {"email_id": "Email #2", "client": "Enterprise Accounts (VP Sales)", "segment": "Enterprise", "amount": f"USD {march_enterprise_usd:,.0f} (March Enterprise invoices)",
     "recommended_action": "Decline", "responsible_owner": "Director of Treasury",
     "policy_justification": "Net 90 exceeds Net 60 (policy 1). Multi-customer extensions and any change to standard terms need written Director of Treasury approval (3b); 'CFO is aware' is not written approval. Offer a case-by-case review.",
     "data_check": "Existing Enterprise terms in the AR file: Net 30 / 45 / 60 only."},
    {"email_id": "Email #3", "client": "Lumen Retail", "segment": lumen_inv['segment'], "amount": f"USD {lumen_inv['amount_due']:,.0f}",
     "recommended_action": "Decline backdating (confirm Net 30 from invoice date)", "responsible_owner": "Billing Ops / Collections Lead",
     "policy_justification": "Invoices carry their actual issue date (policy 2) and Net 30 runs from it, so the due date stays " + f"{lumen_due:%d %b %Y}" + ". Do not re-issue dated 20 Feb. Add an early alert for orders missing a PO.",
     "data_check": f"AR file: INV-19818 issued {lumen_inv['invoice_date']:%d %b %Y}, service start {lumen_inv['service_period_start']:%d %b %Y}, invoice lag {lumen_inv['invoice_lag_days']:.0f} days."},
    {"email_id": "Email #4", "client": "Grupo Varanda", "segment": varanda_inv['segment'], "amount": f"BRL {varanda_inv['amount_due']:,.0f}",
     "recommended_action": "Escalate (suspected payment-redirection fraud)", "responsible_owner": "Treasury + Information Security",
     "policy_justification": "Only Treasury issues or changes bank details (policy 5), so the 9 Mar 'new account' email is not from Nexa. Do not mark paid and do not waive the late fee (only the Collections Lead, after full payment - policy 4). Customer must pay the verified original account.",
     "data_check": f"Bank report: BRL {varanda_row['amount']:,.0f} received {varanda_row['value_date'][:10]} then returned by the bank (beneficiary name mismatch). Invoice open, {varanda_inv['days_overdue']} days overdue."},
    {"email_id": "Email #5", "client": "Tidewater Analytics", "segment": tidewater_inv['segment'], "amount": f"USD {tidewater_inv['amount_due']:,.0f}",
     "recommended_action": "Approve" if tidewater_other.empty else "Request more information", "responsible_owner": "Collections Lead",
     "policy_justification": "One-time extension of 30 days on a single invoice (policy 3a) - new due date " + f"{tidewater_new_due:%d %b %Y}" + ". Confirm in writing and note that no further extensions apply.",
     "data_check": f"Other Tidewater invoices more than 30 days overdue: {len(tidewater_other)}."},
    {"email_id": "Email #6", "client": "Unapplied Cash (3 receipts)", "segment": "Self-Serve / SMB", "amount": "USD 27,600 + GBP 18,250 + EUR 41,000",
     "recommended_action": "Decline", "responsible_owner": "AR Specialist + Sales Ops (Controller if open at close)",
     "policy_justification": "Booking unexplained receipts to miscellaneous income is not allowed. Keep them in suspense, get the signed contracts from Sales Ops, then invoice and apply, or refund if the payer cannot be identified.",
     "data_check": "Receipts found in bank report with no matching invoice: " + ", ".join(
         f"{r['currency']} {r['amount']:,.0f} ({r['value_date'][:10]})" for _, r in unapplied.iterrows()) + "."},
]

df_email_matrix = pd.DataFrame(email_data)

# Presentation
pd.set_option('display.max_colwidth', 120)
display(df_email_matrix[['email_id', 'client', 'recommended_action', 'responsible_owner', 'data_check']])

,email_id,client,recommended_action,responsible_owner,data_check
0,Email #1,Kestrel Media,Approve,AR / Cash Application,"Bank report: wire of 2,400,000 USD on 2026-03-26 (matched by customer + amount to INV-19810)."
1,Email #2,Enterprise Accounts (VP Sales),Decline,Director of Treasury,Existing Enterprise terms in the AR file: Net 30 / 45 / 60 only.
2,Email #3,Lumen Retail,Decline backdating (confirm Net 30 from invoice date),Billing Ops / Collections Lead,"AR file: INV-19818 issued 16 Mar 2026, service start 20 Feb 2026, invoice lag 24 days."
3,Email #4,Grupo Varanda,Escalate (suspected payment-redirection fraud),Treasury + Information Security,"Bank report: BRL 1,850,000 received 2026-03-12 then returned by the bank (beneficiary name mismatch). Invoice open, ..."
4,Email #5,Tidewater Analytics,Approve,Collections Lead,Other Tidewater invoices more than 30 days overdue: 0.
5,Email #6,Unapplied Cash (3 receipts),Decline,AR Specialist + Sales Ops (Controller if open at close),"Receipts found in bank report with no matching invoice: USD 27,600 (2026-03-11), GBP 18,250 (2026-03-20), EUR 41,000..."


**How the AR analysis works**

* **Scope:** customer invoices only (101 of them). PSP payouts are expected cash, not receivables, so they are not part of the aging.
* **Open balance:** invoice amount minus the receipts matched in notebook A, so run notebook A first. A wire returned by the bank does not count as paid. A leftover under 0.5% (bank fees) is treated as closed.
* **Aging:** as of 10 April 2026, counted from the due date, in the buckets Not Due, 1-30, 31-60, 61-90 and 90+ days. Everything is converted to USD at the latest spot rate.
* **Process metrics:** invoice lag (service start to invoice date), payment behaviour (actual vs expected date), unapplied cash, returned wires and overdue share by segment.
* **Emails:** each decision follows Appendix A and is checked against the bank report and AR data (see the `data_check` column).

In [8]:
output_dir = 'analytics_output'
os.makedirs(output_dir, exist_ok=True)

# All invoices with their open balance and USD value
df_ar.to_csv(f'{output_dir}/ar_aging_detailed.csv', index=False)

ar_aging_summary.to_csv(f'{output_dir}/ar_aging_by_bucket_currency.csv', index=False)

segment_performance.to_csv(f'{output_dir}/ar_segment_performance.csv', index=False)

# Process efficiency metrics shown in the dashboard
process_kpis = pd.DataFrame([
    {'metric': 'Open invoices', 'value': len(df_open), 'unit': 'count'},
    {'metric': 'Open AR', 'value': open_usd, 'unit': 'USD'},
    {'metric': 'Overdue share of open AR', 'value': overdue_usd / open_usd * 100, 'unit': '%'},
    {'metric': 'Avg invoice lag after service start', 'value': df_ar['invoice_lag_days'].mean(), 'unit': 'days'},
    {'metric': 'Invoices issued >7 days after service start', 'value': (df_ar['invoice_lag_days'] > 7).mean() * 100, 'unit': '%'},
    {'metric': 'Avg days paid vs expected date', 'value': paid_invoices['days_diff'].mean(), 'unit': 'days'},
    {'metric': 'Invoices paid >3 days late', 'value': (paid_invoices['days_diff'] > 3).mean() * 100, 'unit': '%'},
    {'metric': 'Unapplied receipts', 'value': unapplied['amount_usd'].sum(), 'unit': 'USD'},
    {'metric': 'Unapplied receipts', 'value': len(unapplied), 'unit': 'count'},
])
process_kpis.to_csv(f'{output_dir}/ar_process_kpis.csv', index=False)

df_email_matrix.to_csv(f'{output_dir}/ar_email_decision_matrix.csv', index=False)

print(f"Successfully saved to: {output_dir}")

Successfully saved to: analytics_output
